# 04 — Knowledge Store

This notebook demonstrates the knowledge storage layer for NeuroForge,
including vector storage (ChromaDB) and the NetworkX knowledge graph.

---

## Knowledge Graph

The `KnowledgeGraph` class provides a NetworkX-based directed graph for:
1. **Concept storage** — nodes with attributes (name, definition, difficulty, topics, keywords)
2. **Relationship modeling** — directed edges (prerequisite, related, part_of)
3. **Transitive queries** — find all prerequisites recursively
4. **Topic subgraphs** — filter the graph by topic
5. **Serialization** — save/load graph as JSON
6. **Visualization** — matplotlib rendering with difficulty-based coloring

In [ ]:
import sys
sys.path.insert(0, "..")

from models import Concept, ConceptRelationship, Difficulty
from src.store.knowledge_graph import KnowledgeGraph

print("Imports OK.")

### Build a Sample Knowledge Graph

We'll create concepts for a machine learning curriculum and wire them
together with prerequisite and related relationships.

In [ ]:
# Define concepts
concepts = [
    Concept(
        id="algebra", name="Algebra",
        definition="Branch of mathematics dealing with symbols and rules for manipulating them.",
        topics=["mathematics"], difficulty=Difficulty.EASY,
        keywords=["equations", "variables", "expressions"]
    ),
    Concept(
        id="calculus", name="Calculus",
        definition="Study of continuous change through derivatives and integrals.",
        topics=["mathematics"], difficulty=Difficulty.MEDIUM,
        prerequisites=["algebra"],
        keywords=["derivatives", "integrals", "limits"]
    ),
    Concept(
        id="linear_algebra", name="Linear Algebra",
        definition="Study of linear equations, vector spaces, and transformations.",
        topics=["mathematics", "computer_science"], difficulty=Difficulty.MEDIUM,
        prerequisites=["algebra"],
        keywords=["matrices", "vectors", "eigenvalues"]
    ),
    Concept(
        id="probability", name="Probability",
        definition="Study of random events and their likelihood.",
        topics=["mathematics", "statistics"], difficulty=Difficulty.MEDIUM,
        prerequisites=["calculus"],
        keywords=["distributions", "random variables", "Bayes"]
    ),
    Concept(
        id="machine_learning", name="Machine Learning",
        definition="Algorithms that improve through experience and data.",
        topics=["computer_science", "ai"], difficulty=Difficulty.HARD,
        prerequisites=["linear_algebra", "probability"],
        keywords=["supervised", "unsupervised", "training"]
    ),
    Concept(
        id="deep_learning", name="Deep Learning",
        definition="Neural networks with multiple hidden layers.",
        topics=["ai"], difficulty=Difficulty.HARD,
        prerequisites=["machine_learning"],
        keywords=["CNN", "RNN", "transformer"]
    ),
]

# Define relationships
relationships = [
    ConceptRelationship(source_concept="algebra", target_concept="calculus", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="algebra", target_concept="linear_algebra", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="calculus", target_concept="probability", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="linear_algebra", target_concept="machine_learning", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="probability", target_concept="machine_learning", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="machine_learning", target_concept="deep_learning", relationship_type="prerequisite"),
    ConceptRelationship(source_concept="calculus", target_concept="linear_algebra", relationship_type="related"),
    ConceptRelationship(source_concept="probability", target_concept="linear_algebra", relationship_type="related"),
]

print(f"Defined {len(concepts)} concepts and {len(relationships)} relationships.")

In [ ]:
# Build the graph
kg = KnowledgeGraph()
kg.add_concepts(concepts)
kg.add_relationships(relationships)

print(f"Graph has {kg.graph.number_of_nodes()} nodes and {kg.graph.number_of_edges()} edges.")

### Query Prerequisites

Find all transitive prerequisites for a target concept.

In [ ]:
# What do I need to learn before Machine Learning?
prereqs = kg.get_prerequisites("machine_learning")
print("Prerequisites for Machine Learning:")
for p in prereqs:
    name = kg.graph.nodes[p].get("name", p)
    print(f"  - {name}")

In [ ]:
# What concepts depend on Linear Algebra?
dependents = kg.get_dependents("linear_algebra")
print("Concepts that require Linear Algebra:")
for d in dependents:
    name = kg.graph.nodes[d].get("name", d)
    print(f"  - {name}")

In [ ]:
# What's directly related to Calculus?
related = kg.get_related("calculus")
print("Directly related to Calculus:")
for r in related:
    name = kg.graph.nodes[r].get("name", r)
    print(f"  - {name}")

### Topic Subgraph

Extract only concepts belonging to a specific topic.

In [ ]:
math_graph = kg.get_topic_subgraph("mathematics")
print(f"Mathematics subgraph: {math_graph.number_of_nodes()} nodes, {math_graph.number_of_edges()} edges")
print("Nodes:", list(math_graph.nodes()))

### Visualization

Render the graph with nodes colored by difficulty (green=easy, yellow=medium, red=hard).

In [ ]:
%matplotlib inline
kg.visualize()

### Serialization

Save and reload the graph from disk.

In [ ]:
import tempfile, os
from pathlib import Path

# Save to a temp file
tmp_path = Path(tempfile.mkdtemp()) / "kg_demo.json"
kg.save(str(tmp_path))
print(f"Saved graph to {tmp_path} ({os.path.getsize(tmp_path)} bytes)")

# Load into a new graph
kg2 = KnowledgeGraph()
kg2.load(str(tmp_path))
print(f"Loaded graph: {kg2.graph.number_of_nodes()} nodes, {kg2.graph.number_of_edges()} edges")

# Verify round-trip integrity
assert len(kg2) == len(kg)
print("Round-trip integrity verified.")